Лабораторная работа 1   
Метод Гаусса с выбором главного значения по матрице  
$$
A = 
\begin{bmatrix}
 0,4974 &  0,0000 & -0,1299 &  0,0914 &  0,1523 \\
-0,0305 &  0,3248 &  0,0000 & -0,0619 &  0,0203 \\
 0,0102 & -0,0914 &  0,5887 &  0,0112 &  0,0355 \\
 0,0305 &  0,0000 & -0,0741 &  0,5887 &  0,0000 \\
 0,0203 & -0,0305 &  0,1472 & -0,0122 &  0,4263
\end{bmatrix}
\quad \quad
b = 
\begin{bmatrix}
 1,5875 \\
-1,7590 \\
 1,4139 \\
 1,7702 \\
-2,0767
\end{bmatrix}
$$



In [8]:
import numpy as np

In [9]:
A = np.array([
    [ 0.4974,  0.0000, -0.1299,  0.0914,  0.1523],
    [-0.0305,  0.3248,  0.0000, -0.0619,  0.0203],
    [ 0.0102, -0.0914,  0.5887,  0.0112,  0.0355],
    [ 0.0305,  0.0000, -0.0741,  0.5887,  0.0000],
    [ 0.0203, -0.0305,  0.1472, -0.0122,  0.4263]
])
B = np.array([
    [ 1.5875],
    [-1.7590],
    [ 1.4139],
    [ 1.7702],
    [-2.0767]
])



In [10]:
def find_swap_not_zero(A, k):
    for i in range(k + 1, A.shape[0]):
        if A[i][k] != 0:
            A[[k, i]] = A[[i, k]]
            return

def find_swap_leading_element(A, k, p, swaps):
    max_ = abs(A[k][k])
    ind_i, ind_j = k, k
    for i in range(k, A.shape[0]):
        for j in range(k, A.shape[0]):
            if abs(A[i][j]) > max_:
                ind_i, ind_j = i, j
                max_ = abs(A[i][j])
    if ind_i != k:
        A[[k, ind_i]] = A[[ind_i, k]]
        swaps += 1
    if ind_j != k:
        A[:, [k, ind_j]] = A[:, [ind_j, k]]
        p[k], p[ind_j] = p[ind_j], p[k]
        swaps += 1
    return swaps


def Gauss_right(a_, b_):
    n = b_.shape[0]
    m = a_.shape[1] + b_.shape[1]
    p = np.arange(n)
    A = np.hstack((a_, b_))
    swaps = 0
    for k in range(n - 1):
        swaps = find_swap_leading_element(A, k, p, swaps)
        for i in range(k + 1, n):
            b = A[i][k] / A[k][k]
            for j in range(k + 1, m):
                A[i][j] = A[i][j] - A[k][j] * b
    return A, p, swaps

def Gauss_inv(A, p):
    n, m = A.shape
    x = np.zeros(n) 
    x[n - 1] = A[n - 1][n] / A[n - 1][n - 1]
    for k in range(n - 2, -1, -1):
        sum = 0
        for j in range(k + 1, n):
            sum += A[k][j] * x[j]
        x[k] = (A[k][n] - sum) / A[k][k]
    x_correct = np.zeros(n)
    for i in range(n):
        x_correct[p[i]] = x[i]
    return x_correct

def find_det(A, swaps):
    mul = 1
    for i in range(A.shape[0]):
        mul *= A[i][i]
    return (-1) ** swaps * mul

In [11]:
a = np.array([[ 1.0,  1.0,  1.0], 
              [ 2.0,  3.0,  1.0], 
              [ 1.0, -1.0,  2.0]])

b = np.array([[ 6.0], 
              [11.0], 
              [ 5.0]])


In [12]:
n = A.shape[0]
I = np.eye(n)

G_A, p, swaps = Gauss_right(A, np.hstack((B, I)))
X = Gauss_inv(G_A, p)
print("X:")
np.set_printoptions(formatter={'float_kind': lambda x: f"{x:.5f}"})
print(X)


Ax = np.zeros(n)
for i in range(n):
    total_sum = 0
    for j in range(len(X)):
        total_sum += A[i][j] * X[j]
    Ax[i] = total_sum
r = np.zeros(n)
for i in range(n):
    r[i] = Ax[i] - B[i][0]
print("r:")
np.set_printoptions(formatter={'float_kind': lambda x: f"{x:.5e}"})
print(r)


A_inv = np.zeros((n,n))
for i in range(n):
    G_A[:, [n, n + i + 1]] = G_A[:, [n + i + 1, n]]
    col = Gauss_inv(G_A, p)
    A_inv[:, i] = col
print("Deteminant:")
print(find_det(G_A, swaps))
print("Inverse matrix:")
print(A_inv)

AA = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sum = 0
        for k in range(n):
            sum += A_inv[i][k] * A[k][j]
        AA[i][j] = sum
R = AA - np.eye(n)

print("Невязка по матрице: ")
print(R)

X:
[4.99961 -3.99950 1.99891 2.99954 -6.00005]
r:
[4.44089e-16 -2.22045e-16 -2.22045e-16 0.00000e+00 0.00000e+00]
Deteminant:
0.022848087580690634
Inverse matrix:
[[2.05684e+00 9.71753e-02 6.09001e-01 -3.37084e-01 -7.90170e-01]
 [1.78324e-01 3.09496e+00 1.31449e-01 2.90637e-01 -2.22033e-01]
 [-6.14384e-04 4.74503e-01 1.74207e+00 1.33749e-02 -1.67447e-01]
 [-1.06640e-01 5.46914e-02 1.87724e-01 1.71781e+00 1.98614e-02]
 [-8.80263e-02 5.45249e-02 -6.15756e-01 8.13879e-02 2.42589e+00]]
Невязка по матрице: 
[[2.22045e-16 0.00000e+00 1.38778e-17 -5.20417e-18 5.55112e-17]
 [2.60209e-18 0.00000e+00 -1.38778e-17 -3.03577e-17 -1.38778e-17]
 [-4.33681e-18 1.12757e-17 0.00000e+00 -1.04083e-17 -1.38778e-17]
 [1.08420e-18 5.42101e-19 -3.46945e-18 0.00000e+00 -3.46945e-18]
 [-1.38778e-17 -1.38778e-17 5.55112e-17 -6.93889e-18 0.00000e+00]]
